# SolarEye — Milestones 1 & 2 CPU verification
CSE281, Alamein International University. AI-assisted execution-only notebook. Run after all four reviewed PRs have been integrated. The saved outputs are intentionally empty; execute to generate fresh evidence. Use the development split only. The detailed local Phase 1 evidence notebook remains in the distributed review ZIP.

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

def run(*args):
    """Execute a command, stopping on failure."""
    subprocess.run(list(args), check=True)

root = Path.cwd()
if not (root / "src" / "pipeline.py").exists():
    if (root.parent / "src" / "pipeline.py").exists():
        root = root.parent
    else:
        target = root / "SolarEye"
        if not target.exists():
            run("git", "clone", "https://github.com/mariam123ahmed/SolarEye.git", str(target))
        root = target
os.chdir(root)
sys.path.insert(0, str(root))
assert Path("src/data.py").exists(), "Integrate all member PRs first."
assert Path("src/segmentation.py").exists(), "Integrate all member PRs first."
print("Repository:", root)
run(sys.executable, "-m", "pip", "install", "-r", "requirements.txt")


## Unit tests
The 10 future-stage skips are not implemented M3 tests. Record the actual test count and commit.

In [ ]:
run(sys.executable, "-m", "pytest", "-q")

## Verify real dataset and official split
Download only if needed. Do not redistribute the archive or tune using the held-out test split.

In [ ]:
import json
import pandas as pd
from src.data import build_dataset_manifest, create_dev_test_split

run(sys.executable, "data/download_dataset.py", "--dataset", "msps",
    "--out_dir", "data/raw/msps")
run(sys.executable, "data/validate_dataset.py", "--data_dir", "data/raw/msps",
    "--out_report", "data/validation_report.json")
report = json.loads(Path("data/validation_report.json").read_text())
assert report["status"] == "PASSED"
print(report["counts"])
full = build_dataset_manifest(Path("data/raw/msps"), project_root=Path("."))
dev, held_out = create_dev_test_split(full, preserve_official_split=True)
assert not set(dev["image_id"]) & set(held_out["image_id"])
assert not set(dev["image_sha256"]) & set(held_out["image_sha256"])
print("Development:", len(dev), "Held out:", len(held_out))
Path("data/splits").mkdir(parents=True, exist_ok=True)
dev.to_csv("data/splits/dev_manifest.csv", index=False)
held_out.to_csv("data/splits/test_manifest.csv", index=False)


## Hand-worked metrics (3 × 3)
For TP=2, FP=1 and FN=1: IoU=1/2, Dice=2/3. An intensity error of 3 in one of nine pixels gives MSE=1 and PSNR=20 log10(255).

In [ ]:
import numpy as np
from src.metrics import compute_iou, compute_dice, compute_mse, compute_psnr
pred = np.array([[1, 1, 0], [1, 0, 0], [0, 0, 0]], dtype=np.uint8)
gt = np.array([[1, 0, 0], [1, 1, 0], [0, 0, 0]], dtype=np.uint8)
ref = np.full((3, 3), 100, dtype=np.uint8)
target = ref.copy()
target[0, 0] += 3
assert np.isclose(compute_iou(pred, gt), 0.5)
assert np.isclose(compute_dice(pred, gt), 2 / 3)
assert np.isclose(compute_mse(ref, target), 1)
assert np.isclose(compute_psnr(ref, target), 20 * np.log10(255))
print("IoU:", compute_iou(pred, gt), "Dice:", compute_dice(pred, gt))
print("MSE:", compute_mse(ref, target), "PSNR:", compute_psnr(ref, target))


## Reproduce the development baselines
Run the unmodified baseline algorithms on the whole development split. Avoid appending duplicate archived log entries; save fresh results separately. Record the actual runner and results when submitting.

In [ ]:
from src.config import load_config
from src.evaluate import evaluate_manifest
summaries = []
for name in ("baseline_0", "baseline_1"):
    cfg_path = f"configs/{name}.yaml"
    summaries.append(evaluate_manifest(
        "data/splits/dev_manifest.csv", load_config(cfg_path),
        "experiments/results/reproduction", config_file=cfg_path,
        append_to_log=False,
    ))
summary = pd.DataFrame(summaries)
display(summary[["pipeline", "num_images", "mean_iou", "std_iou",
                 "mean_dice", "total_runtime_s"]])
summary.to_csv("experiments/results/reproduction/summary.csv", index=False)


## Review before submission
Review the development-only EDA in `01_image_analysis.ipynb`, document findings and genuine individual contributions, obtain a student peer review for every PR, and verify the integrated commit. Do not mark M3 or M4 complete.